In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load clean data
X = pd.read_csv('../datasets/credit_X_clean.csv')
y = pd.read_csv('../datasets/credit_y_clean.csv').squeeze()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# Train all 3 models
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# Collect results
results = {}
for name, pred in [
    ('Logistic Regression', lr_pred),
    ('Random Forest', rf_pred),
    ('XGBoost', xgb_pred)
]:
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'F1 Score': round(f1_score(y_test, pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, pred), 4)
    }

results_df = pd.DataFrame(results).T
print("\nResults on Credit Dataset:")
print(results_df)

# Save
results_df.to_csv('../results/credit_results.csv')
print("\n✅ Results saved!")

Train: (24000, 23) | Test: (6000, 23)

Results on Credit Dataset:
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.8078    0.3555   0.6044
Random Forest          0.8120    0.4577   0.6497
XGBoost                0.8118    0.4611   0.6515

✅ Results saved!


In [4]:
import pandas as pd

# Load both result tables
heart_results = pd.read_csv('../results/heart_results.csv', index_col=0)
credit_results = pd.read_csv('../results/credit_results.csv', index_col=0)

print("\n=== COMPARISON ===")

print("\nDataset 1 (Heart):")
print(heart_results)

print("\nDataset 2 (Credit):")
print(credit_results)

print("\nQuick observation:")
print("Best model on Heart:", heart_results['Accuracy'].idxmax())
print("Best model on Credit:", credit_results['Accuracy'].idxmax())



=== COMPARISON ===

Dataset 1 (Heart):
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.5574    0.5450   0.7635
Random Forest          0.5246    0.4759   0.7647
XGBoost                0.4754    0.4808   0.7515

Dataset 2 (Credit):
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.8078    0.3555   0.6044
Random Forest          0.8120    0.4577   0.6497
XGBoost                0.8118    0.4611   0.6515

Quick observation:
Best model on Heart: Logistic Regression
Best model on Credit: Random Forest
